# Caso 01: E-commerce Andino

## Descripción

Una empresa peruana de comercio electrónico vende productos de varias categorías mediante web, app y tiendas físicas. La gerencia necesita analizar clientes, pedidos, vendedores, ventas, descuentos, tiempos de entrega y devoluciones.

### Regla de negocio para los ejercicios

Salvo que la pregunta indique lo contrario, considera como **venta válida** todo pedido cuyo estado sea diferente de `cancelado`. El importe neto de una línea se calcula así:

```sql
cantidad * precio_unitario * (1 - descuento_pct / 100)
```

## Tablas

- `clientes`: información demográfica y fecha de registro.
- `vendedores`: región y meta mensual de cada vendedor.
- `productos`: catálogo, precios y costos.
- `pedidos`: cabecera del pedido, canal, estado y entrega.
- `detalle_pedidos`: productos, cantidades, precios y descuentos.
- `devoluciones`: productos devueltos y montos reembolsados.

## Preguntas básicas (1-10)

1. Muestra todos los registros de la tabla `clientes`.
2. Lista el ID, nombre, categoría y precio de los productos activos.
3. Muestra los pedidos cuyo estado sea `entregado`.
4. Obtén la lista de ciudades únicas de los clientes, ordenadas alfabéticamente.
5. Muestra los 10 productos con mayor precio de lista.
6. Cuenta cuántos pedidos existen en total.
7. Calcula el precio de lista promedio de todos los productos.
8. Muestra los pedidos realizados durante enero de 2026.
9. Cuenta cuántos clientes existen por segmento.
10. Muestra cada línea de pedido incluyendo una columna calculada `importe_neto_linea`.

## Preguntas intermedias (11-30)

11. Muestra cada cliente con su cantidad total de pedidos, incluyendo clientes sin pedidos.
12. Calcula las ventas netas válidas por categoría de producto.
13. Calcula el ticket promedio de los pedidos entregados. Primero suma cada pedido y luego promedia esos totales.
14. Obtén la cantidad de pedidos válidos y las ventas netas por mes.
15. Identifica los 10 clientes con mayor gasto neto en pedidos válidos.
16. Lista los productos que nunca aparecen en `detalle_pedidos`.
17. Lista los clientes que nunca realizaron un pedido.
18. Calcula el promedio de días de entrega por canal para pedidos entregados.
19. Calcula por categoría: unidades vendidas, unidades devueltas y porcentaje de devolución sobre unidades vendidas. Considera ventas válidas.
20. Calcula las ventas mensuales por vendedor y el porcentaje de cumplimiento de su meta mensual.
21. Genera un ranking de productos por ventas netas dentro de cada categoría.
22. Calcula las ventas netas mensuales y el acumulado de ventas desde el primer mes.
23. Calcula el crecimiento porcentual de ventas netas respecto al mes anterior.
24. Para cada cliente con pedidos, muestra la fecha de su primera compra y la fecha de su compra más reciente.
25. Lista los clientes que realizaron pedidos válidos en al menos 3 meses distintos.
26. Muestra los pedidos válidos cuyo importe total sea superior al importe promedio de todos los pedidos válidos.
27. Lista los productos cuyo precio de lista sea superior al precio promedio de su propia categoría.
28. Calcula el porcentaje de participación de cada categoría sobre las ventas netas válidas totales.
29. Identifica el método de pago más utilizado en cada ciudad de envío. Si hay empate, elige alfabéticamente el método de pago.
30. Calcula, por segmento de cliente, el promedio de días transcurridos entre la fecha de registro y la primera compra.

### DDL

In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce;

USE ecommerce;

DROP TABLE IF EXISTS devoluciones;
DROP TABLE IF EXISTS detalle_pedidos;
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS vendedores;
DROP TABLE IF EXISTS clientes;

CREATE OR REPLACE TABLE clientes (
  cliente_id STRING NOT NULL,
  nombre_cliente STRING NOT NULL,
  ciudad STRING NOT NULL,
  region STRING NOT NULL,
  segmento STRING NOT NULL,
  fecha_registro DATE NOT NULL,
  email STRING NOT NULL
) USING DELTA;

CREATE OR REPLACE TABLE vendedores (
  vendedor_id STRING NOT NULL,
  nombre_vendedor STRING NOT NULL,
  region STRING NOT NULL,
  fecha_ingreso DATE NOT NULL,
  meta_mensual DECIMAL(12,2) NOT NULL
) USING DELTA;

CREATE OR REPLACE TABLE productos (
  producto_id STRING NOT NULL,
  nombre_producto STRING NOT NULL,
  categoria STRING NOT NULL,
  marca STRING NOT NULL,
  precio_lista DECIMAL(12,2) NOT NULL,
  costo_unitario DECIMAL(12,2) NOT NULL,
  activo BOOLEAN NOT NULL
) USING DELTA;

CREATE OR REPLACE TABLE pedidos (
  pedido_id STRING NOT NULL,
  cliente_id STRING NOT NULL,
  vendedor_id STRING NOT NULL,
  fecha_pedido DATE NOT NULL,
  canal STRING NOT NULL,
  metodo_pago STRING NOT NULL,
  estado STRING NOT NULL,
  ciudad_envio STRING NOT NULL,
  fecha_entrega DATE
) USING DELTA;

CREATE OR REPLACE TABLE detalle_pedidos (
  pedido_id STRING NOT NULL,
  linea_id INT NOT NULL,
  producto_id STRING NOT NULL,
  cantidad INT NOT NULL,
  precio_unitario DECIMAL(12,2) NOT NULL,
  descuento_pct DECIMAL(5,2) NOT NULL
) USING DELTA;

CREATE OR REPLACE TABLE devoluciones (
  devolucion_id STRING NOT NULL,
  pedido_id STRING NOT NULL,
  producto_id STRING NOT NULL,
  fecha_devolucion DATE NOT NULL,
  cantidad_devuelta INT NOT NULL,
  motivo STRING NOT NULL,
  monto_reembolso DECIMAL(12,2) NOT NULL
) USING DELTA;

### DML

In [0]:
INSERT INTO clientes
SELECT cliente_id, nombre_cliente, ciudad, region, segmento, 
       CAST(fecha_registro AS DATE), email
FROM read_files('/Volumes/workspace/ecommerce/vol_txt/clientes.txt', format => 'csv', header => true);

INSERT INTO vendedores
SELECT vendedor_id, nombre_vendedor, region, 
       CAST(fecha_ingreso AS DATE), 
       CAST(meta_mensual AS DECIMAL(12,2))
FROM read_files('/Volumes/workspace/ecommerce/vol_txt/vendedores.txt', format => 'csv', header => true);

INSERT INTO productos
SELECT producto_id, nombre_producto, categoria, marca, 
       CAST(precio_lista AS DECIMAL(12,2)), 
       CAST(costo_unitario AS DECIMAL(12,2)), 
       CAST(activo AS BOOLEAN)
FROM read_files('/Volumes/workspace/ecommerce/vol_txt/productos.txt', format => 'csv', header => true);

INSERT INTO pedidos
SELECT pedido_id, cliente_id, vendedor_id, 
       CAST(fecha_pedido AS DATE), 
       canal, metodo_pago, estado, ciudad_envio, 
       CAST(fecha_entrega AS DATE)
FROM read_files('/Volumes/workspace/ecommerce/vol_txt/pedidos.txt', format => 'csv', header => true);

INSERT INTO detalle_pedidos
SELECT pedido_id, 
       CAST(linea_id AS INT), 
       producto_id, 
       CAST(cantidad AS INT), 
       CAST(precio_unitario AS DECIMAL(12,2)), 
       CAST(descuento_pct AS DECIMAL(5,2))
FROM read_files('/Volumes/workspace/ecommerce/vol_txt/detalle_pedidos.txt', format => 'csv', header => true);

INSERT INTO devoluciones
SELECT devolucion_id, pedido_id, producto_id, 
       CAST(fecha_devolucion AS DATE), 
       CAST(cantidad_devuelta AS INT), 
       motivo, 
       CAST(monto_reembolso AS DECIMAL(12,2))
FROM read_files('/Volumes/workspace/ecommerce/vol_txt/devoluciones.txt', format => 'csv', header => true);


num_affected_rows,num_inserted_rows
63,63


In [0]:
-- 1. Muestra todos los registros de la tabla clientes.
SELECT *
FROM clientes;

cliente_id,nombre_cliente,ciudad,region,segmento,fecha_registro,email
C0001,Renzo Reyes,Cusco,Sur,Persona,2024-03-06,renzo.reyes1@example.com
C0002,Natalia Aguilar,Piura,Norte,Persona,2024-12-04,natalia.aguilar2@example.com
C0003,Hugo Núñez,Cajamarca,Norte,Empresa,2024-10-12,hugo.nunez3@example.com
C0004,Carla Mendoza,Cusco,Sur,Persona,2025-08-26,carla.mendoza4@example.com
C0005,Carlos Silva,Puno,Sur,Persona,2024-01-19,carlos.silva5@example.com
C0006,Bruno Ortega,Ica,Costa,Persona,2025-11-07,bruno.ortega6@example.com
C0007,Daniela Navarro,Huancayo,Centro,Persona,2024-03-02,daniela.navarro7@example.com
C0008,Gabriela Acosta,Arequipa,Sur,Persona,2025-10-27,gabriela.acosta8@example.com
C0009,Miguel Paredes,Lima,Costa,Persona,2025-11-10,miguel.paredes9@example.com
C0010,Jorge Reyes,Tacna,Sur,Pyme,2025-08-26,jorge.reyes10@example.com


In [0]:
-- 2. Lista el ID, nombre, categoría y precio de los productos activos.
SELECT producto_id, nombre_producto, categoria, precio_lista
FROM productos
WHERE activo = true;

producto_id,nombre_producto,categoria,precio_lista
P0001,Laptop Nova 14,Tecnología,2130.00
P0002,Mouse Ergonómico,Tecnología,1095.00
P0003,Teclado Mecánico,Tecnología,2220.00
P0004,Audífonos Bluetooth,Tecnología,735.00
P0006,Webcam Full HD,Tecnología,2135.00
P0007,SSD Externo 1TB,Tecnología,205.00
P0008,Smartwatch Active,Tecnología,1855.00
P0010,Cafetera Digital,Hogar,145.00
P0011,Freidora de Aire,Hogar,690.00
P0012,Juego de Sábanas,Hogar,165.00


In [0]:
USE ecommerce;

In [0]:
-- 3. Muestra los pedidos cuyo estado sea entregado.
SELECT *
FROM pedidos
WHERE LOWER(estado) = 'entregado'

pedido_id,cliente_id,vendedor_id,fecha_pedido,canal,metodo_pago,estado,ciudad_envio,fecha_entrega
O00001,C0037,V002,2025-08-13,App,Tarjeta,entregado,Cajamarca,2025-08-24
O00003,C0035,V012,2025-09-24,Tienda,Tarjeta,entregado,Puno,2025-09-30
O00004,C0117,V008,2026-05-11,Web,Tarjeta,entregado,Cusco,2026-05-16
O00005,C0057,V010,2025-09-16,App,Efectivo,entregado,Trujillo,2025-09-23
O00006,C0006,V001,2026-01-07,Web,Plin,entregado,Ica,2026-01-14
O00007,C0039,V010,2025-12-08,Web,Tarjeta,entregado,Trujillo,2025-12-18
O00009,C0099,V007,2026-01-11,Web,Transferencia,entregado,Ayacucho,2026-01-14
O00010,C0037,V006,2025-02-13,App,Tarjeta,entregado,Cajamarca,2025-02-15
O00012,C0111,V007,2025-11-28,Web,Transferencia,entregado,Huancayo,2025-12-01
O00013,C0034,V006,2025-02-24,App,Yape,entregado,Trujillo,2025-02-27


In [0]:
-- 4. Obtén la lista de ciudades únicas de los clientes, ordenadas alfabéticamente.
SELECT DISTINCT ciudad
FROM clientes
ORDER BY ciudad;

ciudad
Arequipa
Ayacucho
Cajamarca
Chiclayo
Cusco
Huancayo
Ica
Lima
Piura
Puno


In [0]:
-- 5. Muestra los 10 productos con mayor precio de lista.
SELECT *
FROM productos
ORDER BY precio_lista DESC
LIMIT 10;

-- SELECT *
-- FROM (
    -- SELECT 
    --     *,
    --     row_number() OVER(PARTITION BY categoria ORDER BY precio_lista DESC) ranking
    -- FROM productos
-- )
-- WHERE ranking = 1;



producto_id,nombre_producto,categoria,marca,precio_lista,costo_unitario,activo
P0005,Monitor 24 pulgadas,Tecnología,ViewPlus,2715.00,1746.64,false
P0003,Teclado Mecánico,Tecnología,KeyMax,2220.00,1335.93,true
P0006,Webcam Full HD,Tecnología,VisionX,2135.00,1434.37,true
P0001,Laptop Nova 14,Tecnología,NovaTech,2130.00,1295.55,true
P0008,Smartwatch Active,Tecnología,Pulse,1855.00,959.23,true
P0002,Mouse Ergonómico,Tecnología,ClickPro,1095.00,590.93,true
P0044,Pizarra Magnética,Oficina,IdeaBoard,940.00,590.83,true
P0039,Hub USB-C,Oficina,LinkPro,940.00,607.02,true
P0040,Soporte para Laptop,Oficina,DeskUp,890.00,582.23,true
P0004,Audífonos Bluetooth,Tecnología,SoundGo,735.00,518.89,true


In [0]:
-- 6.Cuenta cuántos pedidos existen en total.
SELECT COUNT(*) AS total_pedidos
FROM pedidos;

total_pedidos
360


In [0]:
-- 7. Calcula el precio de lista promedio de todos los productos.
SELECT AVG(precio_lista) AS precio_promedio
FROM productos;

precio_promedio
543.222222


In [0]:
-- 8. Muestra los pedidos realizados durante enero de 2026.
SELECT *
FROM pedidos
-- WHERE fecha_pedido >= '2026-01-01' AND fecha_pedido < '2026-02-01' -- fecha_pedida <= '2026-01-31'
WHERE fecha_pedido BETWEEN '2026-01-01' AND '2026-01-31';

pedido_id,cliente_id,vendedor_id,fecha_pedido,canal,metodo_pago,estado,ciudad_envio,fecha_entrega
O00006,C0006,V001,2026-01-07,Web,Plin,entregado,Ica,2026-01-14
O00009,C0099,V007,2026-01-11,Web,Transferencia,entregado,Ayacucho,2026-01-14
O00014,C0068,V008,2026-01-17,App,Tarjeta,entregado,Puno,2026-01-24
O00016,C0103,V004,2026-01-09,Web,Plin,entregado,Arequipa,2026-01-16
O00018,C0036,V002,2026-01-22,Tienda,Transferencia,entregado,Cajamarca,2026-01-24
O00019,C0034,V010,2026-01-19,Web,Tarjeta,pendiente,Trujillo,null
O00024,C0080,V008,2026-01-28,App,Plin,cancelado,Puno,null
O00035,C0032,V012,2026-01-17,Web,Plin,cancelado,Arequipa,null
O00048,C0105,V007,2026-01-14,Tienda,Tarjeta,cancelado,Huancayo,null
O00054,C0107,V005,2026-01-15,Web,Tarjeta,entregado,Lima,2026-01-18


In [0]:
-- 9. Cuenta cuántos clientes existen por segmento.
SELECT segmento, COUNT(*) AS total_clientes
FROM clientes
GROUP BY segmento;

segmento,total_clientes
Persona,76
Empresa,10
Pyme,34


In [0]:
-- 10. Muestra cada línea de pedido incluyendo una columna calculada importe_neto_linea.
SELECT 
    *,
    ROUND(cantidad * precio_unitario * (1 - descuento_pct / 100), 2) AS importe_neto_linea
FROM detalle_pedidos;

pedido_id,linea_id,producto_id,cantidad,precio_unitario,descuento_pct,importe_neto_linea
O00001,1,P0034,1,136.57,10.00,122.91
O00001,2,P0024,1,88.29,15.00,75.05
O00001,3,P0012,1,167.53,25.00,125.65
O00001,4,P0031,2,110.51,0.00,221.02
O00002,1,P0004,2,708.06,10.00,1274.51
O00003,1,P0008,2,1807.73,0.00,3615.46
O00003,2,P0029,3,101.84,20.00,244.42
O00003,3,P0002,1,1117.27,0.00,1117.27
O00003,4,P0030,1,150.76,25.00,113.07
O00004,1,P0040,1,920.72,5.00,874.68


In [0]:
select *
from pedidos;

pedido_id,cliente_id,vendedor_id,fecha_pedido,canal,metodo_pago,estado,ciudad_envio,fecha_entrega
O00001,C0037,V002,2025-08-13,App,Tarjeta,entregado,Cajamarca,2025-08-24
O00002,C0092,V007,2026-06-19,App,Plin,pendiente,Ayacucho,null
O00003,C0035,V012,2025-09-24,Tienda,Tarjeta,entregado,Puno,2025-09-30
O00004,C0117,V008,2026-05-11,Web,Tarjeta,entregado,Cusco,2026-05-16
O00005,C0057,V010,2025-09-16,App,Efectivo,entregado,Trujillo,2025-09-23
O00006,C0006,V001,2026-01-07,Web,Plin,entregado,Ica,2026-01-14
O00007,C0039,V010,2025-12-08,Web,Tarjeta,entregado,Trujillo,2025-12-18
O00008,C0111,V007,2025-12-19,Tienda,Yape,pendiente,Huancayo,null
O00009,C0099,V007,2026-01-11,Web,Transferencia,entregado,Ayacucho,2026-01-14
O00010,C0037,V006,2025-02-13,App,Tarjeta,entregado,Cajamarca,2025-02-15


In [0]:
-- 11. Muestra cada cliente con su cantidad total de pedidos, incluyendo clientes sin pedidos.
SELECT 
    c.cliente_id,
    c.nombre_cliente,
    COUNT(*) AS total_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p
ON c.cliente_id = p.cliente_id
GROUP BY c.cliente_id, c.nombre_cliente;

cliente_id,nombre_cliente,total_pedidos
C0036,Sofía Núñez,3
C0067,Miguel Ortega,2
C0107,Marco Medina,3
C0115,Sofía Torres,3
C0066,Sofía Sánchez,1
C0081,Fernando Herrera,1
C0011,Álvaro Silva,7
C0064,José García,2
C0076,Mateo Vargas,6
C0003,Hugo Núñez,4


In [0]:
SELECT *
FROM detalle_pedidos;

pedido_id,linea_id,producto_id,cantidad,precio_unitario,descuento_pct
O00001,1,P0034,1,136.57,10.00
O00001,2,P0024,1,88.29,15.00
O00001,3,P0012,1,167.53,25.00
O00001,4,P0031,2,110.51,0.00
O00002,1,P0004,2,708.06,10.00
O00003,1,P0008,2,1807.73,0.00
O00003,2,P0029,3,101.84,20.00
O00003,3,P0002,1,1117.27,0.00
O00003,4,P0030,1,150.76,25.00
O00004,1,P0040,1,920.72,5.00


In [0]:
SELECT *
FROM productos;

producto_id,nombre_producto,categoria,marca,precio_lista,costo_unitario,activo
P0001,Laptop Nova 14,Tecnología,NovaTech,2130.00,1295.55,true
P0002,Mouse Ergonómico,Tecnología,ClickPro,1095.00,590.93,true
P0003,Teclado Mecánico,Tecnología,KeyMax,2220.00,1335.93,true
P0004,Audífonos Bluetooth,Tecnología,SoundGo,735.00,518.89,true
P0005,Monitor 24 pulgadas,Tecnología,ViewPlus,2715.00,1746.64,false
P0006,Webcam Full HD,Tecnología,VisionX,2135.00,1434.37,true
P0007,SSD Externo 1TB,Tecnología,DataBox,205.00,105.29,true
P0008,Smartwatch Active,Tecnología,Pulse,1855.00,959.23,true
P0009,Licuadora Compacta,Hogar,CasaMix,485.00,329.14,false
P0010,Cafetera Digital,Hogar,Aroma,145.00,102.50,true


In [0]:
-- 12. Calcula las ventas netas válidas por categoría de producto.
SELECT
    pr.categoria,
    ROUND(SUM(dp.cantidad * dp.precio_unitario * (1 - dp.descuento_pct / 100)), 2) AS venta_neto
FROM pedidos AS p
INNER JOIN detalle_pedidos AS dp
ON p.pedido_id = dp.pedido_id
INNER JOIN productos AS pr
ON dp.producto_id = pr.producto_id
WHERE p.estado <> 'cancelado'
GROUP BY pr.categoria

categoria,venta_neto
Deportes,46057.55
Moda,24510.65
Hogar,107762.25
Oficina,125504.25
Belleza,35519.96
Tecnología,484789.88


In [0]:
-- 13. Calcula el ticket promedio de los pedidos entregados. Primero suma cada pedido y luego promedia esos totales.
WITH total_ventas_por_pedido AS (
    SELECT 
        p.pedido_id,
        SUM(dp.cantidad * dp.precio_unitario * (1 - dp.descuento_pct / 100)) AS venta_neto
    FROM pedidos AS p
    INNER JOIN detalle_pedidos AS dp
    ON p.pedido_id = dp.pedido_id
    WHERE estado = 'entregado'
    GROUP BY p.pedido_id
)
SELECT ROUND(AVG(venta_neto), 2) AS promedio_ticket
FROM total_ventas_por_pedido

promedio_ticket
2571.05


In [0]:
-- 14. Obtén la cantidad de pedidos válidos y las ventas netas por mes.
SELECT
    EXTRACT('MONTH', fecha_pedido) AS mes,
    SUM(dp.cantidad * dp.precio_unitario * (1 - dp.descuento_pct / 100)) AS venta_neto,
    COUNT(1) AS cantidad_pedidos_validos
FROM pedidos AS p
INNER JOIN detalle_pedidos AS dp
ON p.pedido_id = dp.pedido_id
WHERE p.estado <> 'cancelado'
GROUP BY EXTRACT('MONTH', fecha_pedido);

mes,venta_neto,cantidad_pedidos_validos
12,49165.93200000,59
5,69411.69150000,94
10,52283.78800000,52
1,96375.95500000,121
3,100307.26250000,110
2,90267.58350000,102
6,70873.34350000,80
9,46879.12600000,58
11,59458.22800000,65
7,48426.42950000,37


In [0]:
-- 15. Identifica los 10 clientes con mayor gasto neto en pedidos válidos.
SELECT
    c.cliente_id,
    c.nombre_cliente,
    SUM(dp.cantidad * dp.precio_unitario * (1 - dp.descuento_pct / 100)) AS gasto_neto
FROM clientes AS c
INNER JOIN pedidos AS p
ON c.cliente_id = p.cliente_id
INNER JOIN detalle_pedidos AS dp
ON p.pedido_id = dp.pedido_id
WHERE p.estado <> 'cancelado'
GROUP BY c.cliente_id, c.nombre_cliente
ORDER BY gasto_neto DESC
LIMIT 10

cliente_id,nombre_cliente,gasto_neto
C0027,Ana Valdez,31382.79700000
C0090,Camila Medina,29452.63000000
C0019,Daniela Peña,26312.57900000
C0096,Gabriela Silva,23669.58500000
C0098,María Acosta,22783.13950000
C0011,Álvaro Silva,21690.86850000
C0034,Alejandra Aguilar,20064.62450000
C0101,Marco Campos,19790.04600000
C0087,Carla Ramírez,16024.25850000
C0076,Mateo Vargas,15415.04600000


In [0]:
DESCRIBE detalle_pedidos

col_name,data_type,comment
pedido_id,string,null
linea_id,int,null
producto_id,string,null
cantidad,int,null
precio_unitario,"decimal(12,2)",null
descuento_pct,"decimal(5,2)",null


In [0]:
-- 16. Lista los productos que nunca aparecen en `detalle_pedidos`.
SELECT *
FROM productos AS p
LEFT ANTI JOIN detalle_pedidos AS dp
ON p.producto_id = dp.producto_id;

producto_id,nombre_producto,categoria,marca,precio_lista,costo_unitario,activo
P0043,Calculadora Científica,Oficina,Numex,530.00,368.61,true
P0044,Pizarra Magnética,Oficina,IdeaBoard,940.00,590.83,true
P0045,Archivador A4,Oficina,OrdenaPro,630.00,400.56,true


In [0]:
SELECT *
FROM productos AS p
WHERE producto_id NOT IN (
    SELECT DISTINCT producto_id
    FROM detalle_pedidos
);

producto_id,nombre_producto,categoria,marca,precio_lista,costo_unitario,activo
P0043,Calculadora Científica,Oficina,Numex,530.00,368.61,true
P0044,Pizarra Magnética,Oficina,IdeaBoard,940.00,590.83,true
P0045,Archivador A4,Oficina,OrdenaPro,630.00,400.56,true


In [0]:
SELECT p.*
FROM productos AS p
LEFT JOIN detalle_pedidos AS d
  ON p.producto_id = d.producto_id
WHERE d.producto_id IS NULL;

producto_id,nombre_producto,categoria,marca,precio_lista,costo_unitario,activo
P0043,Calculadora Científica,Oficina,Numex,530.00,368.61,true
P0044,Pizarra Magnética,Oficina,IdeaBoard,940.00,590.83,true
P0045,Archivador A4,Oficina,OrdenaPro,630.00,400.56,true


In [0]:
DESCRIBE pedidos

col_name,data_type,comment
pedido_id,string,null
cliente_id,string,null
vendedor_id,string,null
fecha_pedido,date,null
canal,string,null
metodo_pago,string,null
estado,string,null
ciudad_envio,string,null
fecha_entrega,date,null


In [0]:
-- 17. Lista los clientes que nunca realizaron un pedido.
SELECT *
FROM clientes AS c
LEFT ANTI JOIN pedidos AS p
ON c.cliente_id = p.cliente_id;

cliente_id,nombre_cliente,ciudad,region,segmento,fecha_registro,email
C0029,Carla Reyes,Chiclayo,Norte,Persona,2026-01-27,carla.reyes29@example.com
C0051,Marco Mendoza,Huancayo,Centro,Persona,2025-12-14,marco.mendoza51@example.com
C0055,Jorge Rojas,Puno,Sur,Pyme,2025-01-26,jorge.rojas55@example.com
C0066,Sofía Sánchez,Chiclayo,Norte,Pyme,2025-05-12,sofia.sanchez66@example.com
C0075,María Espinoza,Huancayo,Centro,Pyme,2025-05-24,maria.espinoza75@example.com
C0081,Fernando Herrera,Ayacucho,Centro,Empresa,2024-11-13,fernando.herrera81@example.com
C0088,Andrea Castro,Tacna,Sur,Persona,2025-07-13,andrea.castro88@example.com
C0113,Natalia Cruz,Cajamarca,Norte,Pyme,2026-01-04,natalia.cruz113@example.com


In [0]:

-- 15. Identifica los 10 clientes con mayor gasto neto en pedidos válidos.
-- 16. Lista los productos que nunca aparecen en `detalle_pedidos`.
-- 17. Lista los clientes que nunca realizaron un pedido.
-- 18. Calcula el promedio de días de entrega por canal para pedidos entregados.
-- 19. Calcula por categoría: unidades vendidas, unidades devueltas y porcentaje de devolución sobre unidades vendidas. Considera ventas válidas.
-- 20. Calcula las ventas mensuales por vendedor y el porcentaje de cumplimiento de su meta mensual.
-- 21. Genera un ranking de productos por ventas netas dentro de cada categoría.
-- 22. Calcula las ventas netas mensuales y el acumulado de ventas desde el primer mes.
-- 23. Calcula el crecimiento porcentual de ventas netas respecto al mes anterior.
-- 24. Para cada cliente con pedidos, muestra la fecha de su primera compra y la fecha de su compra más reciente.
-- 25. Lista los clientes que realizaron pedidos válidos en al menos 3 meses distintos.
-- 26. Muestra los pedidos válidos cuyo importe total sea superior al importe promedio de todos los pedidos válidos.
-- 27. Lista los productos cuyo precio de lista sea superior al precio promedio de su propia categoría.
-- 28. Calcula el porcentaje de participación de cada categoría sobre las ventas netas válidas totales.
-- 29. Identifica el método de pago más utilizado en cada ciudad de envío. Si hay empate, elige alfabéticamente el método de pago.
-- 30. Calcula, por segmento de cliente, el promedio de días transcurridos entre la fecha de registro y la primera compra.